# 🔥 fold3 (distributed) 학습 — Colab T4 (Drive 이어학습)

YouTube 2개 영상을 영상별로 솎아 중복 제거 후 StratifiedKFold(shuffle)로 분산 → 7개 모델 학습.

**중요:** 가중치를 **Google Drive에 저장**하므로 세션이 끊겨도 다시 실행하면 완료분은 건너뛰고 이어감.
**실행 전:** 런타임 → 런타임 유형 변경 → **T4 GPU**
예상 ~5~8시간 (internimage 병목). 끊기면 **모두 실행 다시** 누르면 이어짐.

In [ ]:
# Cell 1: GPU 확인 + 코드 클론
import torch, os
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 — T4로 변경!')
assert torch.cuda.is_available(), 'T4 GPU 런타임으로 변경 후 다시 실행'
!git clone https://github.com/yuntaewon812/fireimage_detection.git /content/fireimage_detection
%cd /content/fireimage_detection

In [ ]:
# Cell 2: Google Drive 연결 → 가중치/결과 영구 저장 (끊겨도 이어학습)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
CKPT = '/content/drive/MyDrive/fireimage_dist_ckpt'
repo = '/content/fireimage_detection'
for sub in ['model_save', 'results']:
    os.makedirs(f'{CKPT}/{sub}', exist_ok=True)
    link = f'{repo}/{sub}'
    if os.path.islink(link):
        os.unlink(link)
    elif os.path.exists(link):
        shutil.rmtree(link, ignore_errors=True)
    os.symlink(f'{CKPT}/{sub}', link)
print('Drive 연결 완료 → 가중치는', CKPT, '에 저장(재접속 시 이어감)')
# 이미 학습된 것 확인
import glob
done = glob.glob(f'{CKPT}/model_save/**/*.pt', recursive=True)
print(f'기존 완료 가중치: {len(done)}개')

In [ ]:
# Cell 3: 패키지 설치
!pip install timm einops transformers yt-dlp kaggle -q
print('설치 완료')

In [ ]:
# Cell 4: Kaggle 인증 (KGAT_ 토큰)
import os, getpass, re
raw = getpass.getpass('Kaggle API Token (KGAT_...): ')
token = re.sub(r'[^A-Za-z0-9_\-]', '', raw)
print('토큰 길이:', len(token), '| 시작:', token[:5])
os.environ['KAGGLE_API_TOKEN'] = token
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
open(os.path.expanduser('~/.kaggle/access_token'), 'w').write(token)
os.chmod(os.path.expanduser('~/.kaggle/access_token'), 0o600)
!kaggle datasets list --user yuntarwon

In [ ]:
# Cell 5: 데이터 다운로드
import os, glob, zipfile
BASE = '/content/fireimage_detection/data/fireimage'
def dl(dataset, dst):
    os.makedirs(dst, exist_ok=True)
    os.system(f'kaggle datasets download {dataset} -p {dst} --unzip')
    for _ in range(2):
        zips = glob.glob(f'{dst}/**/*.zip', recursive=True)
        if not zips: break
        for z in zips:
            try:
                with zipfile.ZipFile(z) as zf: zf.extractall(dst)
                os.remove(z)
            except Exception as e: print('unzip err', e)
dl('yuntarwon/fireimage-abnormal',         f'{BASE}/abnormal')
dl('yuntarwon/fireimage-abnormal-youtube', f'{BASE}/abnormal/youtube')
dl('yuntarwon/fireimage-normal',           f'{BASE}/normal')
imgs=('.jpg','.jpeg','.png','.bmp')
a=sum(1 for f in glob.glob(f'{BASE}/abnormal/**/*',recursive=True) if f.lower().endswith(imgs))
n=sum(1 for f in glob.glob(f'{BASE}/normal/**/*',recursive=True) if f.lower().endswith(imgs))
print(f'normal {n:,} / abnormal {a:,}')

In [ ]:
# Cell 6: video2(eEP8a2u5PbA) 프레임 추출 → youtube2
!python data/youtube_preprocessor.py --url "https://www.youtube.com/watch?v=eEP8a2u5PbA" \
    --subdir youtube2 --sample_every 8 --max_abnormal 600 --max_normal 300
!python data/youtube_preprocessor.py --stats

In [ ]:
# Cell 7: distributed 학습 (7모델×3fold). 끊기면 이 셀부터 다시 → 완료분 자동 스킵
%cd /content/fireimage_detection
!python main_distribute.py --class_name fireimage

In [ ]:
# Cell 8: 결과 확인
import pandas as pd
print(pd.read_csv('/content/fireimage_detection/results/fireimage_dist/metrics.csv').to_string())

In [ ]:
# Cell 9: (선택) 백업 다운로드 — 가중치는 이미 Drive에 있음
import shutil
shutil.make_archive('/content/dist_results', 'zip', '/content/fireimage_detection', 'results/fireimage_dist')
from google.colab import files
files.download('/content/dist_results.zip')